# Google Maps Scraper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mamoo85/open-data-actor-notebooks/blob/main/notebooks/google-maps-places-scraper.ipynb)

**What:** Scrape Google Maps places by keyword and location: name, address, phone, website, rating, review count, hours, categories, coordinates, place URL. Optional website emails and socials, up to 5 reviews. No login. $1.50 per 1,000 places.

**Why:** run the [`google-maps-places-scraper`](https://apify.com/transparent_meteorite/google-maps-places-scraper) Apify actor from Python and analyse the result as a pandas DataFrame, with no scraping code to maintain.

> **Disclosure:** I am the author of this actor. It is pay-per-result; the free plan credit on a new Apify account covers a trial run like this one (this notebook uses a very small sample input).

**Privacy:** Phone numbers and emails are dropped before display; business name, category, rating and website are shown.

## 1. Setup

In [1]:
!pip install -q apify-client pandas

import os, getpass
try:
    from google.colab import userdata
    APIFY_TOKEN = userdata.get('APIFY_TOKEN')          # Colab: add it under the key icon (Secrets)
except Exception:
    APIFY_TOKEN = os.environ.get('APIFY_TOKEN') or getpass.getpass('Apify API token: ')
assert APIFY_TOKEN, 'An Apify API token is required (a free account is enough).'
print('Token loaded.')

Token loaded.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Input
A deliberately small sample so the run is cheap and quick. See the [actor page](https://apify.com/transparent_meteorite/google-maps-places-scraper) for every option.

In [2]:
run_input = {   'searchStringsArray': ['coffee shop'],
    'locationQuery': 'Detroit, MI',
    'maxCrawledPlacesPerSearch': 3}

## 3. Run the actor and read the dataset

In [3]:
from apify_client import ApifyClient

client = ApifyClient(APIFY_TOKEN)
run = client.actor('transparent_meteorite/google-maps-places-scraper').call(run_input=run_input)
# apify-client v2 returns a dict, v3 a Run object; handle both
get = (lambda k, a: run[k]) if isinstance(run, dict) else (lambda k, a: getattr(run, a))
items = list(client.dataset(get('defaultDatasetId', 'default_dataset_id')).iterate_items())
print('Run status:', get('status', 'status'), '| items:', len(items))

[apify.google-maps-places-scraper runId:VJJ6IJsidDzFH5YRx] -> Status: RUNNING, Message: 


[apify.google-maps-places-scraper runId:VJJ6IJsidDzFH5YRx] -> 2026-10-14T07:15:45.682Z ACTOR: Pulling container image of build IypgRpl5anoQRMHgP from registry.


[apify.google-maps-places-scraper runId:VJJ6IJsidDzFH5YRx] -> 2026-10-14T07:15:46.711Z ACTOR: Creating container.


[apify.google-maps-places-scraper runId:VJJ6IJsidDzFH5YRx] -> 2026-10-14T07:15:46.793Z ACTOR: Starting container.


[apify.google-maps-places-scraper runId:VJJ6IJsidDzFH5YRx] -> 2026-10-14T07:15:46.793Z ACTOR: Running under "LIMITED_PERMISSIONS".


...

[apify.google-maps-places-scraper runId:VJJ6IJsidDzFH5YRx] -> 2026-10-14T07:15:54.699Z Done. Delivered 3 places (charged: 3 places, 0 contact enrichments, 0 reviews).


[apify.google-maps-places-scraper runId:VJJ6IJsidDzFH5YRx] -> 2026-10-14T07:15:54.764Z INFO  [Status message]: Done: 3 items


[apify.google-maps-places-scraper runId:VJJ6IJsidDzFH5YRx] -> 2026-10-14T07:15:54.803Z Thanks for using Google Maps Places Scraper. Found a bug or need a field? Open an issue on the Issues tab (https://console.apify.com/actors/djgBif3RPBznx5s4K/issues) with the run URL and we will reply there.


[apify.google-maps-places-scraper runId:VJJ6IJsidDzFH5YRx] -> Status: SUCCEEDED, Message: Done: 3 items


Run status: SUCCEEDED | items: 3


## 4. Tabulate

Phone numbers and emails are dropped before display; business name, category, rating and website are shown.

In [4]:
import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

df = pd.DataFrame(items)

# Privacy: remove contact / personal-data columns before anything is displayed
DROP = ["email", "phone", "author", "registeredagent", "contractor", "officer", "linkedin", "facebook", "instagram", "youtube", "tiktok", "socials", "twitter"]
KEEP = ["emailCount", "phoneCount", "emailProvider"]
df = df[[c for c in df.columns if c in KEEP or not any(p in c.lower() for p in DROP)]]
SHOW = [c for c in ["title", "categoryName", "city", "state", "totalScore", "reviewsCount", "website", "claimedByOwner", "permanentlyClosed", "rank"] if c in df.columns]
df[SHOW]

,title,categoryName,city,state,totalScore,reviewsCount,website,claimedByOwner,permanentlyClosed,rank
0,Saturn Coffee,Coffee shop,Detroit,Michigan,4.8,143,http://www.saturn-coffee.com/,True,False,1
1,Dessert Oasis Coffee Roasters Detroit,Coffee shop,Detroit,Michigan,4.6,1016,http://www.docr.coffee/,True,False,2
2,Madcap Coffee Company,Coffee shop,Detroit,Michigan,4.6,672,http://www.madcapcoffee.com/,True,False,3


## 5. A quick analysis

In [5]:
for c in ['totalScore','reviewsCount']:
    df[c]=pd.to_numeric(df[c],errors='coerce')
print(df['categoryName'].value_counts().head(8))
print('With a website:', int(df['website'].notna().sum()), 'of', len(df))
df.sort_values('reviewsCount',ascending=False)[['title','totalScore','reviewsCount']].head(10)

categoryName
Coffee shop    3
Name: count, dtype: int64
With a website: 3 of 3


,title,totalScore,reviewsCount
1,Dessert Oasis Coffee Roasters Detroit,4.6,1016
2,Madcap Coffee Company,4.6,672
0,Saturn Coffee,4.8,143


## Next steps

- Raise the limits in the input cell to run a larger sample.
- Schedule the actor or call it from your own pipeline: [google-maps-places-scraper on the Apify Store](https://apify.com/transparent_meteorite/google-maps-places-scraper).
- Check the Store page for the current pay-per-result price before scaling up.

Generated by `tools/generate.py`.